#### LangSmith 기본 예제


##### 1) 라이브러리 설치

In [ ]:
# uv add langsmith

##### 2) 환경 변수 설정 (Groq, LangSmith)

- `.env` 파일에 `GROQ_API_KEY`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT` 등록
- `load_dotenv()`가 `.env` 값을 환경 변수로 설정하므로 LangSmith가 별도 코드 없이 읽어서 사용

In [1]:
import os
from dotenv import load_dotenv
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# 필수 환경 변수 확인 (값은 출력하지 않고 설정 여부만 확인)
REQUIRED_ENV = ["GROQ_API_KEY", "LANGSMITH_API_KEY", "LANGSMITH_PROJECT"]
missing = [name for name in REQUIRED_ENV if not os.getenv(name)]
if missing:
    raise RuntimeError(f".env 파일에 다음 환경 변수가 없습니다: {', '.join(missing)}")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# LANGSMITH_TRACING 이 'true' 여야 트레이스가 LangSmith 로 전송됨
print("LangSmith 트레이싱:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

LangSmith 트레이싱: true
LangSmith 프로젝트: MyLangchainProject


##### LangSmith와 LangChain을 활용한 기본 로깅 예제

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langsmith import traceable

# LangSmith 설정(LANGSMITH_TRACING / API_KEY / PROJECT / ENDPOINT)은
# 위에서 load_dotenv() 로 환경 변수에 이미 로드되어 있으므로 다시 대입할 필요가 없음

# LLM 모델 설정 (Groq 사용: OpenAI 호환 API)
llm = ChatOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    model="openai/gpt-oss-120b",  
    temperature=0.7
)

# LangSmith로 실행 추적
@traceable(run_type="chain", name="Simple_Chain")
def ask_question(question: str):

    # 개별 메시지 템플릿 정의
    system_message = SystemMessagePromptTemplate.from_template(
        "당신은 유용한 AI 비서입니다."
    )
    user_message = HumanMessagePromptTemplate.from_template(
        "{question}"
    )

    chat_prompt = ChatPromptTemplate.from_messages([
        system_message,
        user_message,
    ])
    
    messages = chat_prompt.format_messages(question=question)
    response = llm.invoke(messages)

    return response.content

# 테스트 실행
question = "LangGraph와 LangChain의 차이점은 무엇인가요?"
answer = ask_question(question)

# 결과 출력
print("\n🔹 [AI 답변]:")
print(answer)



🔹 [AI 답변]:
## LangGraph와 LangChain의 차이점

| 구분 | **LangChain** | **LangGraph** |
|------|---------------|---------------|
| **핵심 목표** | LLM(대형 언어 모델)을 **프롬프트, 체인, 에이전트** 형태로 쉽게 연결·조합하여 **애플리케이션**을 만들기 위한 프레임워크 | LLM을 **노드와 엣지**로 구성된 **그래프** 형태로 모델링하고, **동적 흐름 제어·상태 관리**를 제공하는 **워크플로 엔진** |
| **주요 개념** | • **Chain** (시퀀스) <br>• **Agent** (도구 호출) <br>• **Tool** (외부 API) <br>• **Memory** (대화 기록) | • **Node** (작업 단위) <br>• **Edge** (전이 조건) <br>• **State** (전역/노드 로컬 상태) <br>• **Graph Scheduler** (실행 엔진) |
| **구조** | 트리/선형 혹은 간단한 DAG 형태의 파이프라인이 기본. 복잡한 흐름은 직접 코드로 제어. | **명시적인 그래프**(Directed Acyclic 혹은 cyclic)로 정의. 노드 간 전이 로직을 선언적으로 기술. |
| **워크플로 제어** | `run()` 호출 시 순차/조건부 로직을 파이썬 코드로 구현. | 그래프 스키마에 따라 **자동 스케줄링**·**조건부 전이**가 수행. 상태 변화에 따라 흐름이 동적으로 바뀜. |
| **상태·컨텍스트 관리** | `Memory` 객체를 사용해 대화 히스토리 등 하나의 전역 메모리 관리. | **State 객체**를 노드별·전역적으로 선언 가능. 그래프 실행 중에 상태를 읽고 쓰는 것이 기본 동작. |
| **도구·플러그인** | `Tool` 인터페이스를 통해 외부 API, DB, 검색 엔진 등과 연결. | `Node` 자체가 도구 역할을 하며, **다중 입력/출력**과 **조건부 전이**를 통해 복합 도